<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/ch12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementing convolutional layers with PyTorch

In [1]:
import numpy as np
import torch
from sklearn.datasets import load_sample_images

In [2]:
sample_images = np.stack(load_sample_images()["images"])
# convert NumPy array to torch tensor and rescale the pixel values from 0-255 to 0-1
sample_images = torch.tensor(sample_images, dtype=torch.float32)/255

In [3]:
load_sample_images()["images"][0].shape

(427, 640, 3)

In [4]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [5]:
# PyTorch wants channel dimension before the height and width dimensions
sample_images_permuted = sample_images.permute(0,3,1,2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [6]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70,120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [7]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels = 3, out_channels= 32, kernel_size =7)
fmaps = conv_layer(cropped_images)

In [8]:
fmaps.shape

torch.Size([2, 32, 64, 114])

In [9]:
import torch.nn.functional as F

class DepthPool(torch.nn.Module):
  def __init__(self, kernel_size, stride=None, padding =0):
    super().__init__()
    self.kernel_size = kernel_size
    self.stride = stride if stride is not None else kernel_size
    self.padding = padding

  def forward(self, inputs):
    batch, channels, height, width = inputs.shape
    Z=inputs.reshape(batch, channels, height*width) # merge spatial dimensions
    Z = Z.permute (0,2,1) # switch spatial dimension with channels
    Z = F.max_pool1d(Z, kernel_size= self.kernel_size,
                     stride = self.stride,
                     padding = self.padding)
    Z = Z.permute(0,2,1) # switch spatial dimensions back
    return Z.reshape(batch, -1, height, width)


In [10]:
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [11]:
# let's take the depthwise max pooling across the 3 channels
depth_pooled_images = DepthPool(kernel_size=3, padding=0)(cropped_images)
depth_pooled_images.shape

torch.Size([2, 1, 70, 120])

In [12]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)
output.shape

torch.Size([2, 3, 1, 1])

In [13]:
# a depthwise separable convolutional layer

class SeparableConv2d(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
    super().__init__()
    self.depthwise_conv = nn.Conv2d(
        in_channels = in_channels,
        out_channels = in_channels, # one output channel per input channel
        kernel_size = kernel_size,
        stride = stride,
        padding = padding,
        groups = in_channels #one filter per input channel
    )
    self.pointwise_conv = nn.Conv2d(
        in_channels = in_channels,
        out_channels = out_channels,
        kernel_size = 1,
        stride = 1,
        padding = 0
    )
  def forward (sefl, inputs):
    return self.pointwise_conv(self.depthwise_conv(inputs))

## Let's build ResNet-34 from scratch

In [14]:
# first, let's create a residual unit
from functools import partial
class ResidualUnit(nn.Module):
  def __init__(self, in_channels, out_channels, stride=1):
    super().__init__()
    DeafultConv2d = partial(nn.Conv2d, kernel_size=3, stride=1, padding=1, bias=False)
    self.main_layers= nn. Sequential(
        DefaultConv2d(in_channels, out_channels, stride=stride),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        DefaultConv2d(out_channels, out_channels),
        nn.BatchNorm2d(out_channels)
    )
    if stride > 1:
      self.skip_connection = nn.Sequential(
          DefaultConv2d(in_channels, out_channels, kernel_size=1, stride=stride, padding =0), #kernel size =1 no matter what we set the size
          nn.BatchNorm2d(out_channels)
      )
    else:
      self.skip_connection = nn.Identity()

  def forward(self, inputs):
    return F.relu(self.main_layers(inputs) + self.skip_connection(inputs)) ## warning in_channels=out_channels otherwise this does not make sense.

In [15]:
class ResNet34(nn.Module):
  def __init__(self):
    super().__init__()
    layers = [
        nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False),
        nn.BatchNorm2d(num_features=64),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
    ]
    prev_filters=64
    for filters in [64]*3 + [128]*4 + [256]*6 + [512]*3: #[64, 64, 64, 128, 128, ....]
      stride = 1 if filters ==prev_filters else 2
      layers.append(ResidualUnit(in_channels=prev_filters, out_channels=filters, stride=stride))
      prev_filters = filters
    layers += [
        nn.AdaptiveAvgPool2d(output_size = 1),
        nn.Flatten(),
        nn.LazyLinear(10) #we do not have to specify the input
    ]
    self.layers = nn.Sequential(*layers)

  def forward(self, inputs):
    return self.layers(inputs)



## Load ConvNeXt using TorchVision

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [17]:
weights = torchvision.models.ConvNeXt_Base_Weights.IMAGENET1K_V1
model = torchvision.models.convnext_base(weights=weights).to(device)

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:01<00:00, 188MB/s]


In [18]:
transforms = weights.transforms() #pre-process the images for this model
preprocessed_images = transforms(sample_images_permuted)

In [19]:
model.eval() # switch to evaluation mode
with torch.no_grad(): #
  y_logits = model(preprocessed_images.to(device))

In [20]:
y_logits.shape

torch.Size([2, 1000])

In [21]:
y_pred=torch.argmax(y_logits, dim=1)
y_pred

tensor([698, 985], device='cuda:0')

In [22]:
# what are these classes?
class_names=weights.meta["categories"]
[class_names[i] for i in y_pred]

['palace', 'daisy']

In [23]:
y_top5_logits, y_top5_class_ids = y_logits.topk(k=5, dim=1)
[class_names[i] for i in y_top5_class_ids[0]], [class_names[i] for i in y_top5_class_ids[1]]

(['palace', 'monastery', 'lakeside', 'bell cote', 'boathouse'],
 ['daisy', 'pot', 'ant', 'honeycomb', 'vase'])

In [24]:
y_top5_logits.softmax(dim=1)

tensor([[0.8486, 0.1167, 0.0194, 0.0089, 0.0065],
        [0.7015, 0.0834, 0.0805, 0.0758, 0.0588]], device='cuda:0')

## Transfer Learning

In [27]:
from google.colab import drive


drive.mount('/content/drive')
root = "/content/drive/MyDrive/flowers102"

DefaultFlowers102 = partial(torchvision.datasets.Flowers102, root = "root",
                            transform = weights.transforms(), download = True) # partial because we do not specify the split
train_set = DefaultFlowers102(split = "train")
test_set = DefaultFlowers102(split = "test")
valid_set = DefaultFlowers102(split = "val")



Mounted at /content/drive


100%|██████████| 345M/345M [00:21<00:00, 15.7MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.65MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 11.7MB/s]


In [28]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)
valid_loader = DataLoader(valid_set, batch_size=32, shuffle=False)

In [ ]:
class_names = ['pink primrose', 'hard-leaved pocket orchid', 'canterbury bells', 'sweet pea', 'english marigold', 'tiger lily', 'moon orchid', 'bird of paradise', 'monkshood', 'globe thistle', 'snapdragon', "colt's foot", 'king protea', 'spear thistle', 'yellow iris', 'globe-flower', 'purple coneflower', 'peruvian lily', 'balloon flower', 'giant white arum lily', 'fire lily', 'pincushion flower', 'fritillary', 'red ginger', 'grape hyacinth', 'corn poppy', 'prince of wales feathers', 'stemless gentian', 'artichoke', 'sweet william', 'carnation', 'garden phlox', 'love in the mist', 'mexican aster', 'alpine sea holly', 'ruby-lipped cattleya', 'cape flower', 'great masterwort', 'siam tulip', 'lenten rose', 'barbeton daisy', 'daffodil', 'sword lily', 'poinsettia', 'bolero deep blue', 'wallflower', 'marigold', 'buttercup', 'oxeye daisy', 'common dandelion', 'petunia', 'wild pansy', 'primula', 'sunflower', 'pelargonium', 'bishop of llandaff', 'gaura', 'geranium', 'orange dahlia', 'pink-yellow dahlia?', 'cautleya spicata', 'japanese anemone', 'black-eyed susan', 'silverbush', 'californian poppy', 'osteospermum', 'spring crocus', 'bearded iris', 'windflower', 'tree poppy', 'gazania', 'azalea', 'water lily', 'rose', 'thorn apple', 'morning glory', 'passion flower', 'lotus', 'toad lily', 'anthurium', 'frangipani', 'clematis', 'hibiscus', 'columbine', 'desert-rose', 'tree mallow', 'magnolia', 'cyclamen', 'watercress', 'canna lily', 'hippeastrum', 'bee balm', 'ball moss', 'foxglove', 'bougainvillea', 'camellia', 'mallow', 'mexican petunia', 'bromelia', 'blanket flower', 'trumpet creeper', 'blackberry lily']

In [30]:
[name for name, child in model.named_children()]

['features', 'avgpool', 'classifier']

In [32]:
model.classifier # head of the model

Sequential(
  (0): LayerNorm2d((1024,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=1024, out_features=1000, bias=True)
)

In [33]:
# let's change the last linear layer
n_classes = 102
model.classifier[2] = nn.Linear(in_features=1024, out_features=n_classes).to(device)

In [34]:
# freeze all the parameters in the model
for param in model.parameters():
  param.requires_grad = False

# unfreeze the parameters in the last linear layer
for param in model.classifier.parameters():
  param.requires_grad = True


In [35]:
pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 48.5 MB/s eta 0:00:00


In [36]:
# This evaluation function checks how our model performs on a given dataset
# From notes of Chapter 10
import torchmetrics

def evaluate_tm(model, data_loader, metric):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch) #update at each iteration
  return metric.compute()

In [37]:
import time
def train_with_early_stopping(model, optimizer, criterion, metric,
          train_loader, valid_loader, n_epochs,
            patience=10, checkpoint_path=None, scheduler = None):
  checkpoint_path = checkpoint_path or "my_checkpoint.pt"

  history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
  best_metric = 0.0 #we save the current best validation metric
  patience_counter = 0

  for epoch in range(n_epochs):

    # We MUST force the model back into training mode at the start of every epoch.
    model.train()

    # We wipe the metric memory clean before the new epoch starts.
    metric.reset()

    total_loss = 0
    t0=time.time()

    for X_batch, y_batch in train_loader:

      # Send data to GPU/CPU memory
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      # Forward pass: get model predictions
      y_pred = model(X_batch)

      # Calculate how wrong the model is (Loss value)
      loss = criterion(y_pred, y_batch)

      # We use '.item()' to extract the raw Python number from the loss tensor.
      total_loss += loss.item()

      # Backpropagation: Calculate gradients and update model parameters
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

      # We feed the current batch's predictions and targets into the tracker.
      metric.update(y_pred, y_batch)

    # Calculate average loss for this training epoch
    mean_loss = total_loss / len(train_loader)
    history["train_losses"].append(mean_loss)
    train_metric = metric.compute().item()
    valid_metric = evaluate_tm(model, valid_loader, metric).item()
    if valid_metric > best_metric:
          torch.save(model.state_dict(), checkpoint_path)
          best_metric = valid_metric
          best = " (best)" #print (best) next to the validation metric if better
          patience_counter = 0
    else:
          patience_counter += 1
          best = ""

    t1 = time.time()
    history["train_metrics"].append(train_metric)
    history["valid_metrics"].append(valid_metric)
    print(f"Epoch {epoch + 1}/{n_epochs}, "
          f"train loss: {history['train_losses'][-1]:.4f}, "
          f"train metric: {history['train_metrics'][-1]:.4f}, "
          f"valid metric: {history['valid_metrics'][-1]:.4f}{best}"
          f" in {t1 - t0:.1f}s"
        )
    if scheduler is not None:
      # change the learning rate according to the scheduler's rule
          scheduler.step()
    if patience_counter >= patience:
            print("Early stopping!")
            break
  # Reload the highest-accuracy weights so the model doesn't
  # stick with the worse, overfitted weights from the final epoch.
  # Because we are 10 epochs past the best model

  model.load_state_dict(torch.load(checkpoint_path))
  return history

In [39]:
optimizer = torch.optim.NAdam(model.parameters(), lr=0.002)
criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes).to(device) #n_classes = 102
n_epochs = 20
history = train_with_early_stopping(model, optimizer, criterion, accuracy, train_loader, valid_loader, n_epochs)

Epoch 1/20, train loss: 3.9445, train metric: 0.2520, valid metric: 0.6245 (best) in 30.4s
Epoch 2/20, train loss: 2.0305, train metric: 0.7637, valid metric: 0.8069 (best) in 29.6s
Epoch 3/20, train loss: 0.9583, train metric: 0.9137, valid metric: 0.8510 (best) in 30.9s
Epoch 4/20, train loss: 0.5346, train metric: 0.9569, valid metric: 0.8784 (best) in 32.7s
Epoch 5/20, train loss: 0.3287, train metric: 0.9794, valid metric: 0.8765 in 30.0s
Epoch 6/20, train loss: 0.2239, train metric: 0.9863, valid metric: 0.8843 (best) in 30.8s
Epoch 7/20, train loss: 0.1616, train metric: 0.9912, valid metric: 0.8892 (best) in 31.0s
Epoch 8/20, train loss: 0.1173, train metric: 0.9922, valid metric: 0.8980 (best) in 31.6s
Epoch 9/20, train loss: 0.1010, train metric: 0.9941, valid metric: 0.8961 in 30.6s
Epoch 10/20, train loss: 0.0719, train metric: 0.9971, valid metric: 0.8951 in 30.2s
Epoch 11/20, train loss: 0.0682, train metric: 0.9971, valid metric: 0.9000 (best) in 30.8s
Epoch 12/20, train

In [42]:
import torchvision.transforms.v2 as T

## data augmentation function and normalization instead of using weights.transform()
transforms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomResizedCrop(size=(224,224), scale=(0.8,1.0)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [43]:
root2 = "/content/drive/MyDrive/flowers102_augmented"

DefaultFlowers102_augmented = partial(torchvision.datasets.Flowers102, root = "root2",
                            transform = transforms, download = True) # partial because we do not specify the split
train_set_augmented = DefaultFlowers102_augmented(split = "train")


100%|██████████| 345M/345M [00:41<00:00, 8.29MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.85MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 50.0MB/s]


In [44]:
# let's look at other layers in the model
model.features

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((128,), eps=1e-06, elementwise_affine=True)
  )
  (1): Sequential(
    (0): CNBlock(
      (block): Sequential(
        (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
        (1): Permute()
        (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=128, out_features=512, bias=True)
        (4): GELU(approximate='none')
        (5): Linear(in_features=512, out_features=128, bias=True)
        (6): Permute()
      )
      (stochastic_depth): StochasticDepth(p=0.0, mode=row)
    )
    (1): CNBlock(
      (block): Sequential(
        (0): Conv2d(128, 128, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=128)
        (1): Permute()
        (2): LayerNorm((128,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=128, out_features=512, bias=True)
        (4): GELU(approx

In [46]:
# unfreeze the whole model
for param in model.parameters():
  param.requires_grad = True

In [47]:
optimizer = torch.optim.NAdam(model.parameters(), lr=0.0002) # new optimizer with lr = 1/10 from before
# criterion and accuracy are the same as before
n_epochs = 5
history = train_with_early_stopping(model, optimizer, criterion, accuracy, train_loader, valid_loader, n_epochs)

Epoch 1/5, train loss: 0.0515, train metric: 0.9863, valid metric: 0.8961 (best) in 61.1s
Epoch 2/5, train loss: 0.0343, train metric: 0.9863, valid metric: 0.9137 (best) in 56.6s
Epoch 3/5, train loss: 0.0420, train metric: 0.9902, valid metric: 0.8912 in 56.1s
Epoch 4/5, train loss: 0.0475, train metric: 0.9843, valid metric: 0.9245 (best) in 58.9s
Epoch 5/5, train loss: 0.0297, train metric: 0.9931, valid metric: 0.9353 (best) in 57.2s
